# Timonel SFT v3 — Reentrenamiento con prompt limpio (sin regla de decision)**Objetivo:** reentrenar Llama 3.1 8B con el prompt limpio.**Contexto:**- CC003 v1 tenia el prompt con 'terminar: Valida SOLO si dist < 15m'.- Eso es REGLA DE DECISION, no contrato. Mide obediencia, no discriminacion.- El baseline trivial (if dist<15: terminar else ir_a) saco 100%.- Ahora el prompt dice solo 'terminar: declara la mision como cumplida'.- El modelo debe inferir por si mismo cuando terminar no aplica.**Referencia:** EXPEDIENTE/89 seccion 14.

## 1. Instalar dependencias

In [ ]:
!pip install -q unsloth!pip install -q --no-deps trl peft accelerate bitsandbytes!pip install -q datasets

## 2. Cargar modelo base con QLoRA

In [ ]:
from unsloth import FastLanguageModelimport torchmax_seq_length = 2048dtype = Noneload_in_4bit = Truemodel, tokenizer = FastLanguageModel.from_pretrained(    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",    max_seq_length = max_seq_length,    dtype = dtype,    load_in_4bit = load_in_4bit,)

## 3. Configurar LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(    model,    r = 16,    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",                      "gate_proj", "up_proj", "down_proj"],    lora_alpha = 32,    lora_dropout = 0.0,    bias = "none",    use_gradient_checkpointing = "unsloth",    random_state = 3407,    use_rslora = False,    loftq_config = None,)

## 4. Cargar dataset con prompt limpio

In [ ]:
from datasets import load_datasetimport jsonTRAIN_PATH = "/kaggle/input/datasets/ivnmielniczuk/timonel-cc003/train_cc003.jsonl"EVAL_PATH = "/kaggle/input/datasets/ivnmielniczuk/timonel-cc003/eval_cc003_confirmatorio.jsonl"dataset_train = load_dataset("json", data_files=TRAIN_PATH, split="train")eval_raw = [json.loads(l) for l in open(EVAL_PATH)]print(f"Train: {len(dataset_train)} | Eval (confirmatorio): {len(eval_raw)}")

## 5. VERIFICACION — el prompt NO debe tener regla de decision

In [ ]:
# Regla 15 (memo 89 seccion 14): verificar que el prompt NO contenga# regla de decision antes de entrenar. Aborta si encuentra la regla.REGLAS_PROHIBIDAS = [    "Valida SOLO",    "dist < 15m",    "desvio > 30",    "si estas a menos",    "si estas mal alineado",]print("=== VERIFICANDO PROMPTS DE ENTRENAMIENTO ===")errores = 0for i, ej in enumerate(dataset_train):    user_content = ej["messages"][1]["content"]    for regla in REGLAS_PROHIBIDAS:        if regla in user_content:            print(f"  [FAIL] Ejemplo {i} contiene: '{regla}'")            errores += 1print()if errores > 0:    raise ValueError(f"ABORTADO: {errores} reglas de decision encontradas en el prompt. Corregir antes de entrenar.")print(f"  [OK] {len(dataset_train)} ejemplos de train limpios")print()print("=== VERIFICANDO PROMPTS DE EVALUACION ===")errores_eval = 0for i, caso in enumerate(eval_raw):    prompt = caso.get("prompt_limpio", "")    for regla in REGLAS_PROHIBIDAS:        if regla in prompt:            print(f"  [FAIL] Eval {i} contiene: '{regla}'")            errores_eval += 1if errores_eval > 0:    raise ValueError(f"ABORTADO: {errores_eval} reglas de decision en eval. Corregir antes de evaluar.")print(f"  [OK] {len(eval_raw)} casos de eval limpios")print()print(">>> PROMPTS VERIFICADOS. Se puede entrenar. <<<")

## 6. Aplicar chat template

In [ ]:
from unsloth import get_chat_templatetokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")def formatear(ejemplo):    texto = tokenizer.apply_chat_template(        ejemplo["messages"],        tokenize=False,        add_generation_prompt=False,    )    return {"text": texto}dataset_train = dataset_train.map(formatear)print("=== Ejemplo formateado ===")print(dataset_train[0]["text"][:800])

## 7. Configurar y correr el entrenamiento

In [ ]:
from trl import SFTTrainerfrom transformers import TrainingArgumentsfrom unsloth import is_bfloat16_supportedtrainer = SFTTrainer(    model = model,    tokenizer = tokenizer,    train_dataset = dataset_train,    dataset_text_field = "text",    max_seq_length = max_seq_length,    dataset_num_proc = 2,    packing = False,    args = TrainingArguments(        per_device_train_batch_size = 2,        gradient_accumulation_steps = 4,        warmup_steps = 3,        max_steps = 60,        learning_rate = 2e-4,        fp16 = not is_bfloat16_supported(),        bf16 = is_bfloat16_supported(),        logging_steps = 5,        optim = "adamw_8bit",        weight_decay = 0.01,        lr_scheduler_type = "linear",        seed = 3407,        output_dir = "outputs_cc003_v3",        report_to = "none",    ),)stats = trainer.train()print(stats)

## 8. Guardar el adaptador LoRA v3

In [ ]:
model.save_pretrained("timonel_lora_cc003_v3")tokenizer.save_pretrained("timonel_lora_cc003_v3")print("Adaptador guardado en timonel_lora_cc003_v3/")!ls -la timonel_lora_cc003_v3/

## 9. Evaluar en confirmatorio CC003 (con prompt_limpio)

In [ ]:
import refrom collections import CounterFastLanguageModel.for_inference(model)def extraer_json(texto):    i = texto.find("{")    j = texto.rfind("}")    if i == -1 or j == -1 or j <= i:        return None    try:        return json.loads(texto[i:j+1])    except Exception:        return Nonedef predecir(prompt_user):    messages = [        {"role": "system", "content": "Sos un agente de navegacion. Respondes EXCLUSIVAMENTE con JSON valido."},        {"role": "user", "content": prompt_user},    ]    inputs = tokenizer.apply_chat_template(        messages,        tokenize=True,        add_generation_prompt=True,        return_tensors="pt",    ).to("cuda")    with torch.no_grad():        out = model.generate(            input_ids=inputs,            max_new_tokens=64,            temperature=0.0,            do_sample=False,            use_cache=True,        )    respuesta = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)    return respuestaaciertos = 0total = 0uso = Counter()por_accion_ok = Counter()por_accion_tot = Counter()errores = []import math as _mfor caso in eval_raw:    state = caso["state"]    dx = state["meta_x"] - state["pos_x"]    dz = state["meta_z"] - state["pos_z"]    dist = _m.sqrt(dx*dx + dz*dz)    prompt_user = caso["prompt_limpio"]    crudo = predecir(prompt_user)    parsed = extraer_json(crudo)    pred = parsed.get("accion") if parsed else "_error_"    true = caso["accion_optima"]    uso[pred] += 1    por_accion_tot[true] += 1    if pred == true:        aciertos += 1        por_accion_ok[true] += 1    else:        errores.append({"true": true, "pred": pred, "dist": round(dist,1), "familia": caso["familia"]})    total += 1print(f"=== RESULTADO VALIDACION CC003 (prompt limpio) ===")print(f"Accuracy: {aciertos}/{total} = {100*aciertos/total:.1f}%")print(f"Uso de acciones: {dict(uso)}")print(f"Por accion: {dict(por_accion_ok)} / {dict(por_accion_tot)}")print()print(f"Errores: {len(errores)}")for e in errores[:15]:    print(f"  true={e['true']} pred={e['pred']} dist={e['dist']}m familia={e['familia']}")print()print("=== COMPARACION CON BASELINE TRIVIAL ===")print("Baseline trivial (if dist<15: terminar else ir_a):")print("  76/76 = 100.0% (calculado local)")